In [ ]:
pip install trl==0.11.3

In [2]:
# Cell 1: Cài đặt thư viện (giữ nguyên)
!pip install transformers[torch] datasets accelerate evaluate trl peft bitsandbytes rouge_score scipy tqdm -q
!pip install protobuf==4.25.3 -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 19.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.3 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.3 which is incompatible.
ydf 0.13.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.3 which is incompatible.


In [3]:
# Cell 2: Import và chuẩn bị dữ liệu (giữ nguyên)
import torch
import warnings
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq
)
from peft import PeftModel
from trl import PPOTrainer, PPOConfig
from trl.models.modeling_value_head import AutoModelForSeq2SeqLMWithValueHead
import evaluate
import numpy as np
from tqdm import tqdm

warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Đang sử dụng thiết bị: {device}")

MODEL_NAME = "VietAI/vit5-base"
MAX_SAMPLES = 20000
MAX_LENGTH = 512

OUTPUT_SFT_DIR = "sft_adapter_summarization"
OUTPUT_GRPO_DIR = "grpo_adapter_summarization"  # Đổi tên thư mục output

dataset = load_dataset("nam194/vietnews", split=f"train[:{MAX_SAMPLES}]")

# Chuẩn hóa (Giống hệt SFT)
def preprocess_data(example):
    example["input_text"] = "tóm tắt: " + example["article"]
    example["target_text"] = example["abstract"]
    return example

dataset = dataset.map(
    preprocess_data,
    remove_columns=["guid", "title", "abstract", "article"]
)

# Xáo trộn và Lấy tập train
dataset = dataset.shuffle(seed=42)
split_datasets = dataset.train_test_split(test_size=0.1, seed=42)
grpo_run_dataset = split_datasets["train"] # 18000 mẫu

print(f"Đã tải {len(grpo_run_dataset)} mẫu để chạy GRPO.")

Đang sử dụng thiết bị: cuda


README.md:   0%|          | 0.00/748 [00:00<?, ?B/s]

data/train-00000-of-00001-84acb79f6c6547(…):   0%|          | 0.00/170M [00:00<?, ?B/s]

data/validation-00000-of-00001-210cc51bf(…):   0%|          | 0.00/38.3M [00:00<?, ?B/s]

data/test-00000-of-00001-123f98d55067eb7(…):   0%|          | 0.00/38.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/99134 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/22184 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/22498 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Đã tải 18000 mẫu để chạy GRPO.


In [26]:
# Cell 3: Giải nén và load SFT từ archive (SỬA LỖI QUANTIZATION)
print("--- GIẢI NÉN VÀ LOAD SFT TỪ ARCHIVE ---")

import os
import zipfile
import shutil
from transformers import BitsAndBytesConfig

# Tên file archive
SFT_ARCHIVE = "sft_adapter_summary_archive.zip"

# Cấu hình quantization
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True
)

# Kiểm tra xem file archive có tồn tại không
if not os.path.exists(SFT_ARCHIVE):
    print(f"Lỗi: File {SFT_ARCHIVE} không tồn tại!")
    print("Hãy đảm bảo file được upload đúng cách.")
else:
    print(f"Đã tìm thấy file {SFT_ARCHIVE}")

    # Tạo thư mục output nếu chưa tồn tại
    if not os.path.exists(OUTPUT_SFT_DIR):
        os.makedirs(OUTPUT_SFT_DIR)
        print(f"Đã tạo thư mục {OUTPUT_SFT_DIR}")

    # Giải nén file zip
    print("Đang giải nén archive...")
    with zipfile.ZipFile(SFT_ARCHIVE, 'r') as zip_ref:
        zip_ref.extractall(OUTPUT_SFT_DIR)

    print("✓ Giải nén hoàn tất")

    # Kiểm tra các file đã giải nén
    extracted_files = os.listdir(OUTPUT_SFT_DIR)
    print(f"Các file trong {OUTPUT_SFT_DIR}:")
    for file in extracted_files:
        print(f"  - {file}")

    # Load base model với quantization config mới
    print("Đang load base model...")
    base_model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quantization_config,  # SỬA: dùng quantization_config
        device_map="auto",
        torch_dtype=torch.float16
    )

    # Load SFT adapter từ thư mục đã giải nén
    print("Đang load SFT adapter...")
    sft_model = PeftModel.from_pretrained(base_model, OUTPUT_SFT_DIR)

    print("✓ Đã load thành công mô hình SFT từ archive")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("✓ Đã load tokenizer")

print("--- LOAD SFT TỪ ARCHIVE HOÀN TẤT ---")

--- GIẢI NÉN VÀ LOAD SFT TỪ ARCHIVE ---
Đã tìm thấy file sft_adapter_summary_archive.zip
Đang giải nén archive...
✓ Giải nén hoàn tất
Các file trong sft_adapter_summarization:
  - README.md
  - tokenizer.json
  - adapter_model.safetensors
  - tokenizer_config.json
  - spiece.model
  - special_tokens_map.json
  - adapter_config.json
  - checkpoint-3000
  - training_args.bin
Đang load base model...
Đang load SFT adapter...
✓ Đã load thành công mô hình SFT từ archive
✓ Đã load tokenizer
--- LOAD SFT TỪ ARCHIVE HOÀN TẤT ---


In [59]:
# Cell 4: Custom GRPO Trainer (FIXED - STABLE VERSION)
import torch
import torch.nn.functional as F
from trl import PPOTrainer
from typing import Optional, Dict, List, Tuple
import numpy as np

class GRPOTrainer(PPOTrainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.step_count = 0

    def compute_simple_value(self, rewards, response_lengths, device):
        values = []
        for reward, length in zip(rewards, response_lengths):
            # FIX: Scale value theo reward mới (100)
            value = torch.full((length,), reward.item() * 0.8, device=device)
            values.append(value)
        return values

    def grpo_loss(
        self,
        logprobs: torch.Tensor,
        old_logprobs: torch.Tensor,
        advantages: torch.Tensor,
        kl_penalty: float = 0.0001  # GIẢM THÊM 10x vì reward lớn hơn 10x
    ) -> Tuple[torch.Tensor, Dict[str, float]]:
        """
        GRPO loss function - SCALE CHO REWARD 100
        """
        # Scale advantages cho reward lớn
        advantages = advantages / 10.0  # Normalize advantages

        ratio = torch.exp(logprobs - old_logprobs)
        clipped_ratio = torch.clamp(ratio, 0.8, 1.2)

        # Clamp advantages cho reward lớn
        advantages = torch.clamp(advantages, -10.0, 10.0)  # Tăng clamp

        policy_loss = -torch.mean(torch.min(ratio * advantages, clipped_ratio * advantages))

        kl_div = torch.mean(old_logprobs - logprobs)
        kl_penalty_loss = kl_penalty * kl_div

        total_loss = policy_loss + kl_penalty_loss

        stats = {
            "grpo/loss/total": total_loss.item(),
            "grpo/loss/policy": policy_loss.item(),
            "grpo/loss/kl_penalty": kl_penalty_loss.item(),
            "grpo/advantages_mean": advantages.mean().item(),
            "grpo/kl_div": kl_div.item(),
            "grpo/ratio_mean": ratio.mean().item(),
        }

        return total_loss, stats

    def step(
        self,
        queries: List[torch.Tensor],
        responses: List[torch.Tensor],
        scores: List[torch.Tensor],
        response_masks: Optional[List[torch.Tensor]] = None,
    ):
        """
        GRPO training step - STABLE VERSION
        """
        current_device = self.current_device

        # 1. Pad queries (encoder inputs)
        padded_queries = torch.nn.utils.rnn.pad_sequence(
            queries,
            batch_first=True,
            padding_value=self.tokenizer.pad_token_id
        ).to(current_device)

        query_masks = torch.nn.utils.rnn.pad_sequence(
            [torch.ones_like(q) for q in queries],
            batch_first=True,
            padding_value=0
        ).to(current_device)

        # 2. Pad responses (decoder inputs)
        padded_responses = torch.nn.utils.rnn.pad_sequence(
            responses,
            batch_first=True,
            padding_value=self.tokenizer.pad_token_id
        ).to(current_device)

        response_masks = torch.nn.utils.rnn.pad_sequence(
            [torch.ones_like(r) for r in responses],
            batch_first=True,
            padding_value=0
        ).to(current_device)

        # 3. Model inputs
        model_input_dict = {
            "input_ids": padded_queries,
            "attention_mask": query_masks,
            "decoder_input_ids": padded_responses,
            "decoder_attention_mask": response_masks
        }

        # 4. Get OLD logprobs (no grad)
        with torch.no_grad():
            model_outputs = self.model(**model_input_dict)
            logits = model_outputs[0].detach()

        # 5. Calculate OLD logprobs và values
        response_logprobs = []
        response_values = []

        response_lengths = [len(r) for r in responses]
        rewards_tensor = torch.stack(scores)
        values_list = self.compute_simple_value(rewards_tensor, response_lengths, current_device)

        for i, response in enumerate(responses):
            response_length = len(response)
            response_logits = logits[i, :response_length]
            logprobs = F.log_softmax(response_logits, dim=-1)
            token_logprobs = logprobs[torch.arange(response_length, device=current_device), response]
            response_logprobs.append(token_logprobs.sum())
            response_value = values_list[i]
            response_values.append(response_value.mean())

        old_logprobs = torch.stack(response_logprobs)
        values = torch.stack(response_values)
        rewards = torch.stack(scores)

        # 6. Compute advantages (nên dương)
        advantages = rewards - values.detach()

        # 7. Get NEW logprobs (with grad)
        torch.cuda.empty_cache()

        new_model_outputs = self.model(**model_input_dict)
        new_logits = new_model_outputs[0]

        new_response_logprobs = []
        for i, response in enumerate(responses):
            response_length = len(response)
            new_response_logits = new_logits[i, :response_length]
            new_logprobs = F.log_softmax(new_response_logits, dim=-1)
            new_token_logprobs = new_logprobs[torch.arange(len(response), device=current_device), response]
            new_response_logprobs.append(new_token_logprobs.sum())

        new_logprobs = torch.stack(new_response_logprobs)

        # 8. Compute GRPO loss (STABLE VERSION)
        loss, stats = self.grpo_loss(
            new_logprobs,
            old_logprobs,
            advantages,
            kl_penalty=0.001  # RẤT NHỎ
        )

        # 9. Gradient clipping và backward
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=0.5)
        loss = loss / self.config.gradient_accumulation_steps
        self.accelerator.backward(loss)

        # 10. Optimizer step
        if (self.step_count + 1) % self.config.gradient_accumulation_steps == 0:
            self.optimizer.step()
            self.optimizer.zero_grad()

        if self.lr_scheduler is not None:
            self.lr_scheduler.step()

        self.step_count += 1

        # Add reward stats
        stats.update({
            "grpo/mean_reward": rewards.mean().item(),
            "grpo/mean_value": values.mean().item(),
            "grpo/logprob_diff": (new_logprobs - old_logprobs).mean().item(),
        })

        # Clean up
        del model_outputs, new_model_outputs, logits, new_logits
        torch.cuda.empty_cache()

        return stats

In [60]:
# Cell 5: Chuẩn bị model và tokenizer cho GRPO (SỬA QUANTIZATION)
print("--- CHUẨN BỊ HUẤN LUYỆN GRPO ---")

# Sử dụng PPOConfig làm base
grpo_config = PPOConfig(
    learning_rate=1.41e-5,
    batch_size=64,
    mini_batch_size=8,
    gradient_accumulation_steps=8,
    remove_unused_columns=False,
    log_with=None,
    tracker_project_name=None,
    optimize_cuda_cache=True,
    is_encoder_decoder=True,
    seed=42,
)

# Sử dụng SFT model đã load từ archive làm base cho GRPO
grpo_peft_model = PeftModel.from_pretrained(base_model, OUTPUT_SFT_DIR, is_trainable=True)
grpo_model = AutoModelForSeq2SeqLMWithValueHead(grpo_peft_model)
grpo_model.is_peft_model = True

# Reference model (không trainable) - cũng từ SFT
ref_base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,  # SỬA: dùng quantization_config
    device_map="auto",
    torch_dtype=torch.float16
)
ref_peft_model = PeftModel.from_pretrained(ref_base_model, OUTPUT_SFT_DIR, is_trainable=False)
ref_model = AutoModelForSeq2SeqLMWithValueHead(ref_peft_model)
ref_model.is_peft_model = True
for param in ref_model.parameters():
    param.requires_grad = False

# Khởi tạo GRPO Trainer
grpo_trainer = GRPOTrainer(
    config=grpo_config,
    model=grpo_model,
    ref_model=ref_model,
    tokenizer=tokenizer,
    dataset=grpo_run_dataset,
    data_collator=None
)

# Reward function
rouge_metric = evaluate.load("rouge")

def compute_rouge_reward(predictions, references):
    """
    Tính ROUGE-L F-measure làm phần thưởng.
    """
    try:
        rouge_scores = rouge_metric.compute(
            predictions=predictions,
            references=references,
            use_aggregator=False
        )
        key = 'rougeL' if 'rougeL' in rouge_scores else 'rougeLsum'

        # SCALE LÊN 100
        rewards = [score * 100.0 for score in rouge_scores[key]]
        return rewards
    except Exception as e:
        print(f"Lỗi khi tính ROUGE: {e}. Trả về reward 0.")
        return [0.0] * len(predictions)

print("--- KHỞI TẠO GRPO TRAINER HOÀN TẤT ---")

--- CHUẨN BỊ HUẤN LUYỆN GRPO ---
--- KHỞI TẠO GRPO TRAINER HOÀN TẤT ---


In [61]:
# Cell 5.5: Thêm hàm tính value đơn giản (nếu cần)
def compute_simple_value(rewards, response_lengths, device):
    """Tính values đơn giản dựa trên rewards"""
    values = []
    for reward, length in zip(rewards, response_lengths):
        # Value là reward trung bình cho toàn bộ sequence
        value = torch.full((length,), reward.item(), device=device)
        values.append(value)
    return values

In [62]:
# Cell 6: Huấn luyện GRPO (ĐÃ TỐI ƯU MEMORY)
print(f"Bắt đầu huấn luyện GRPO với {len(grpo_run_dataset)} mẫu...")
print(f"Batch size: {grpo_config.batch_size}, Tích lũy: {grpo_config.gradient_accumulation_steps}")

# Giảm hơn nữa để tiết kiệm memory
generation_kwargs = {
    "max_new_tokens": 64,  # GIẢM XUỐNG 64 tokens
    "num_beams": 1,
    "no_repeat_ngram_size": 2,
    "pad_token_id": tokenizer.pad_token_id,
    "eos_token_id": tokenizer.eos_token_id,
    "do_sample": False,
    "early_stopping": True,
}

# Giảm batch size trong dataloader
grpo_trainer.dataloader = torch.utils.data.DataLoader(
    grpo_run_dataset,
    batch_size=32,  # GIẢM batch size
    shuffle=True,
    collate_fn=lambda x: x
)

# Sử dụng tqdm để theo dõi tiến độ
total_batches = len(grpo_trainer.dataloader)
pbar = tqdm(grpo_trainer.dataloader, desc=f"GRPO Training", total=total_batches)

successful_batches = 0

for batch_idx, batch in enumerate(pbar):
    try:
        # batch chứa các cột từ grpo_run_dataset
        prompt_texts = [item['input_text'] for item in batch]
        reference_texts = [item['target_text'] for item in batch]

        # 1. Tokenize prompts
        prompt_tensors = tokenizer(
            prompt_texts,
            padding=True,
            truncation=True,
            max_length=256,  # GIẢM max_length
            return_tensors="pt"
        ).to(grpo_trainer.current_device)

        query_tensors_ids = prompt_tensors['input_ids']

        # 2. Tạo tóm tắt (responses) từ mô hình Policy
        with torch.no_grad():
            response_tensors = grpo_trainer.model.generate(
                input_ids=query_tensors_ids,
                attention_mask=prompt_tensors['attention_mask'],
                **generation_kwargs
            )

        # 3. Decode tóm tắt (responses)
        response_texts = tokenizer.batch_decode(response_tensors, skip_special_tokens=True)

        # 4. Tính phần thưởng (rewards)
        rewards_list = compute_rouge_reward(response_texts, reference_texts)

        # Đảm bảo rewards trên cùng device với model
        rewards_tensors = [torch.tensor(r, device=grpo_trainer.current_device, dtype=torch.float32) for r in rewards_list]

        # Đảm bảo tất cả tensor trên cùng device
        list_query_tensors = [q.to(grpo_trainer.current_device) for q in prompt_tensors['input_ids']]
        list_response_tensors = [r.to(grpo_trainer.current_device) for r in response_tensors]

        if not rewards_tensors or len(rewards_tensors) == 0:
            print(f"Batch {batch_idx}: rewards_tensors bị rỗng. Bỏ qua bước này.")
            continue

        # Clear cache trước khi step
        torch.cuda.empty_cache()

        # Sử dụng GRPO step
        stats = grpo_trainer.step(list_query_tensors, list_response_tensors, rewards_tensors)

        mean_reward = torch.stack(rewards_tensors).mean().item()
        successful_batches += 1

        pbar.set_postfix({
            "mean_reward": f"{mean_reward:.2f}",
            "grpo_loss": f"{stats.get('grpo/loss/total', 0):.2f}",
            "success": f"{successful_batches}/{batch_idx+1}",
            "mem_alloc": f"{torch.cuda.memory_allocated()//1024**2}MB"
        })

        # Log thông tin mỗi 5 batch
        if batch_idx % 5 == 0:
            print(f"\nBatch {batch_idx}:")
            print(f"  Mean Reward: {mean_reward:.2f}")
            print(f"  GRPO Loss: {stats.get('grpo/loss/total', 0):.4f}")
            print(f"  GPU Memory: {torch.cuda.memory_allocated()//1024**2}MB / {torch.cuda.memory_reserved()//1024**2}MB")

        # Clean up
        del prompt_tensors, response_tensors, rewards_tensors
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"Lỗi trong batch {batch_idx}: {e}")
        import traceback
        traceback.print_exc()
        torch.cuda.empty_cache()
        continue

print(f"--- GRPO TRAINING HOÀN TẤT ---")
print(f"Đã xử lý thành công {successful_batches}/{total_batches} batch")
print("✓ Huấn luyện GRPO hoàn tất!")

Bắt đầu huấn luyện GRPO với 18000 mẫu...
Batch size: 64, Tích lũy: 8


GRPO Training:   0%|          | 1/563 [00:09<1:24:50,  9.06s/it, mean_reward=38.26, grpo_loss=-0.60, success=1/1, mem_alloc=1209MB]


Batch 0:
  Mean Reward: 38.26
  GRPO Loss: -0.6048
  GPU Memory: 1209MB / 6468MB


GRPO Training:   1%|          | 6/563 [00:52<1:20:25,  8.66s/it, mean_reward=33.58, grpo_loss=-0.41, success=6/6, mem_alloc=1209MB]


Batch 5:
  Mean Reward: 33.58
  GRPO Loss: -0.4051
  GPU Memory: 1209MB / 6468MB


...


Batch 555:
  Mean Reward: 31.78
  GRPO Loss: -0.4709
  GPU Memory: 1223MB / 6698MB


GRPO Training: 100%|█████████▉| 561/563 [1:21:38<00:17,  8.70s/it, mean_reward=30.05, grpo_loss=-0.49, success=561/561, mem_alloc=1223MB]


Batch 560:
  Mean Reward: 30.05
  GRPO Loss: -0.4866
  GPU Memory: 1223MB / 6476MB


GRPO Training: 100%|██████████| 563/563 [1:21:54<00:00,  8.73s/it, mean_reward=29.93, grpo_loss=-0.49, success=563/563, mem_alloc=1223MB]

--- GRPO TRAINING HOÀN TẤT ---
Đã xử lý thành công 563/563 batch
✓ Huấn luyện GRPO hoàn tất!


In [63]:
grpo_model.pretrained_model.save_pretrained(OUTPUT_GRPO_DIR)

In [66]:
import evaluate
import torch
from datasets import load_dataset
from tqdm import tqdm

def evaluate_grpo_rouge_l():
    """
    Đánh giá điểm ROUGE-L của mô hình GRPO
    """
    print("🔍 Đang đánh giá ROUGE-L cho GRPO...")

    # Tải tập test
    test_dataset = load_dataset("nam194/vietnews", split="test[:200]")

    # Chuẩn bị metric
    rouge_metric = evaluate.load("rouge")

    all_predictions = []
    all_references = []

    # Chuyển mô hình sang chế độ đánh giá
    grpo_model.eval()

    print("🔄 Đang tạo tóm tắt...")
    for i in tqdm(range(len(test_dataset))):
        example = test_dataset[i]
        input_text = "tóm tắt: " + example["article"]
        reference_text = example["abstract"]

        # Tokenize
        inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).to(device)

        # Generate
        with torch.no_grad():
            outputs = grpo_model.generate(
                **inputs,
                max_new_tokens=128,
                num_beams=1,
                no_repeat_ngram_size=2,
                early_stopping=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)
        all_predictions.append(prediction)
        all_references.append(reference_text)

    # Tính ROUGE scores
    rouge_scores = rouge_metric.compute(
        predictions=all_predictions,
        references=all_references,
        use_aggregator=True
    )

    # Hiển thị kết quả - ROUGE-L
    print("\n📊 KẾT QUẢ ĐÁNH GIÁ GRPO:")
    print(f"🎯 ROUGE-L: {rouge_scores['rougeL'] * 100:.2f}")
    return rouge_scores['rougeL']

rouge_l_score = evaluate_grpo_rouge_l()


🔍 Đang đánh giá ROUGE-L cho GRPO...
🔄 Đang tạo tóm tắt...


100%|██████████| 200/200 [33:13<00:00,  9.97s/it]



📊 KẾT QUẢ ĐÁNH GIÁ GRPO:
🎯 ROUGE-L: 27.23


In [67]:
def DemoGRPO(prompt_text):
    """
    Chạy demo tóm tắt CHỈ DÙNG mô hình GRPO đã load (Base + Adapter).
    """
    if tokenizer is None or grpo_model is None:
        print("LỖI: Tokenizer hoặc Model GRPO chưa được load. Vui lòng chạy lại các ô code trước.")
        return None

    print(f"Input: {prompt_text}")
    full_prompt = "tóm tắt: " + prompt_text

    # Sửa lỗi device - lấy device từ pretrained_model
    device = grpo_model.pretrained_model.device

    inputs = tokenizer(full_prompt, return_tensors="pt", max_length=512, truncation=True).to(device)

    print(f"--- Đang chạy demo với mô hình: GRPO ---")

    with torch.no_grad():
        outputs = grpo_model.generate(
            **inputs,
            max_new_tokens=256,
            num_beams=1,
            no_repeat_ngram_size=2,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Output (GRPO): {result}\n" + "-"*20)
    return result

# Test demo
test_prompts = [
   "Chào các bạn, mình tên là Mai Anh, năm nay mình 9 tuổi và hiện đang học lớp 3. Mình sống cùng gia đình ở một ngôi nhà nhỏ gần công viên. Mình rất thích học Toán và Tiếng Việt, nhưng môn vẽ thì mình cũng rất yêu thích. Ngoài học, mình thích chơi cầu lông và đọc sách, nhất là truyện cổ tích. Mình có một chú mèo tên là Miu, nó rất dễ thương và hay chơi đùa với mình. Mình luôn cố gắng học tốt để sau này có thể trở thành bác sĩ giúp đỡ mọi người. Hy vọng sẽ kết bạn được với nhiều bạn trong lớp!",
    "Gia đình của em có nuôi một chú chó. Tên của chú là Vàng. Vì chú có một bộ lông màu vàng. Vàng nặng khoảng bốn ki-lô-gam. Cái đầu tròn như quả bưởi. Hai chiếc tai hình tam giác. Chiếc mũi màu đen rất thính. Đôi mắt to tròn như hạt nhãn. Cái miệng với hàm răng bé xíu. Em yêu quý Vàng."
,
    "Theo Trung tâm Dự báo KTTV quốc gia, ngày 15/11, ở khu vực Bắc bộ và Bắc Trung bộ không mưa, sáng sớm có sương mù và sương mù nhẹ rải rác, ngày nắng. Đêm và sáng sớm trời lạnh. Khu vực Tây Nguyên và Nam bộ ngày nắng, chiều tối và đêm có mưa giông. Khu vực từ Thừa Thiên Huế đến Phú Yên và Tây Nguyên có mưa vừa, mưa to, cục bộ có nơi mưa rất to và dông. Trong mưa dông có khả năng xảy ra lốc,sét và gió giật mạnh. Đề phòng mưa lớn có khả năng gây ra tình trạng ngập úng tại các vùng trũng, thấp; lũ quét trên các sông, suối nhỏ, sạt lở đất trên sườn dốc"
]

print("🧪 DEMO GRPO MODEL:\n")
for i, prompt in enumerate(test_prompts, 1):
    print(f"Demo {i}:")
    DemoGRPO(prompt)
    print()

🧪 DEMO GRPO MODEL:

Demo 1:
Input: Chào các bạn, mình tên là Mai Anh, năm nay mình 9 tuổi và hiện đang học lớp 3. Mình sống cùng gia đình ở một ngôi nhà nhỏ gần công viên. Mình rất thích học Toán và Tiếng Việt, nhưng môn vẽ thì mình cũng rất yêu thích. Ngoài học, mình thích chơi cầu lông và đọc sách, nhất là truyện cổ tích. Mình có một chú mèo tên là Miu, nó rất dễ thương và hay chơi đùa với mình. Mình luôn cố gắng học tốt để sau này có thể trở thành bác sĩ giúp đỡ mọi người. Hy vọng sẽ kết bạn được với nhiều bạn trong lớp!
--- Đang chạy demo với mô hình: GRPO ---
Output (GRPO): * mình tên Mai Anh, năm nay 9 tuổi và hiện đang học lớp 3. Mình tên là Mai_ Mai, hiện mình đang ở một ngôi nhà nhỏ gần công viên. Hiện mình sống cùng gia đình ở ngôi_ nhà ở gần khu công_. Bạn có một chú mèo tên Miu, mèo rất dễ thương và thích chơi cầu lông và đọc sách.
--------------------

Demo 2:
Input: Gia đình của em có nuôi một chú chó. Tên của chú là Vàng. Vì chú có một bộ lông màu vàng. Vàng nặng khoảng 